### README

Identify disease-related variants (pathogenic/likely pathogenic) annottated in ClinVar database located at the +1 position of NAGNAG (2nd N).

*Output coordinates at base 1
**Input data clinvar.vcf.gz is not shared, but can be downloaded from https://www.ncbi.nlm.nih.gov/clinvar/. For this analysis, the data used was donwloaded on July 17th, 2025.

### Requirements

In [ ]:
# Install
"""
pandas==2.2.3
"""

In [ ]:
# Import libraries
import pandas as pd
import re


### Constants

In [ ]:
# Input: ClinVar data
CLINVAR_PATH = '/PATH/TO/clinvar.vcf.gz'

# Input: annotation data
GTF_PATH = '/PATH/TO/Homo_sapiens.GRCh38.111.gtf.gz'

# Output: set directory
OUTPUT_DIR = '/PATH/TO/OUTPUT'

### Functions

In [ ]:
# EXTRACT UPSTREAM AND DOWNSTREAM EXONS COORDINATES FROM GTF FILE

def extract_exon_pairs(gtf_df: pd.DataFrame) -> pd.DataFrame:
   
    # Filter only exon entries
    exon_df = gtf_df[gtf_df['Type'] == 'exon'].copy()

    #replace , by ; in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(',',';')
    #replace : by = in Attributes column
    exon_df['Attributes'] = exon_df['Attributes'].str.replace(':','=')

    # Extract parent mRNA (transcript_id)
    exon_df['mRNA'] = (
        exon_df['Attributes']
        .str.split('transcript_id "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Extract parent gene (gene_id)
    exon_df['Gene'] = (
        exon_df['Attributes']
        .str.split('gene_name "', n=1, expand=True)[1]
        .str.split('"', n=1, expand=True)[0]
    )

    # Sort by gene, transcript, and genomic start
    exon_df = exon_df.sort_values(by=['Gene', 'mRNA', 'Start'])

    # Assign coordinates of the downstream exon ("below")
    exon_df['Start_Below'] = exon_df.groupby('mRNA')['Start'].shift(-1)
    exon_df['End_Below'] = exon_df.groupby('mRNA')['End'].shift(-1)

    # Drop rows where downstream exon doesn't exist
    exon_df = exon_df.dropna(subset=['Start_Below', 'End_Below']).copy()

    # Keep relevant columns and remove duplicates
    exon_df = exon_df[['SeqID', 'Start', 'End', 'Start_Below', 'End_Below', 'Gene', 'STR']]
    exon_df = exon_df.drop_duplicates()
    

    return exon_df

In [ ]:
# EXTRACT ALL ANNOTATED NAGNAG FROM GTF FILE

def extract_nagnag(exon_df: pd.DataFrame) -> pd.DataFrame:
    
    df = exon_df.copy()

    df['Acceptor'] = df.apply(lambda row: row['Start_Below'] if row['STR'] == '+' else row['End'], axis=1)

    results = []

    for (chr_, strand_, gene_), group in df.groupby(['SeqID', 'STR', 'Gene']):
        
        group = group.sort_values('Acceptor').reset_index(drop=True)

        for i in range(1, len(group)):
            
            acc1 = group.loc[i-1, 'Acceptor']
            acc2 = group.loc[i, 'Acceptor']
            
            diff = acc2 - acc1
            
            if abs(diff) == 3:
                
                if strand_ == '+':
                    IS_1 = group.loc[i-1, 'End']+1
                    IS_2 = group.loc[i, 'End']+1
                    shortIE = acc1-1
                    longIE = acc2-1

                    results.append({
                        'SeqID': chr_,
                        'Gene': gene_,
                        'STR': strand_,
                        'shortIS': IS_1,
                        'shortIE': shortIE,
                        'longIS': IS_1,
                        'longIE': longIE
                    })

                    if IS_1 != IS_2:
                        results.append({
                            'SeqID': chr_,
                            'Gene': gene_,
                            'STR': strand_,
                            'shortIS': IS_2,
                            'shortIE': shortIE,
                            'longIS': IS_2,
                            'longIE': longIE
                        })


                else:
                    IE_1 = group.loc[i-1, 'Start_Below']-1
                    IE_2 = group.loc[i, 'Start_Below']-1
                    shortIS = acc2+1
                    longIS = acc1+1

                    results.append({
                        'SeqID': chr_,
                        'Gene': gene_,
                        'STR': strand_,
                        'shortIS': shortIS,
                        'shortIE': IE_1,
                        'longIS': longIS,
                        'longIE': IE_1
                    })  

                    if IE_1 != IE_2:
                        results.append({
                            'SeqID': chr_,
                            'Gene': gene_,
                            'STR': strand_,
                            'shortIS': shortIS,
                            'shortIE': IE_2,
                            'longIS': longIS,
                            'longIE': IE_2
                        })

    return pd.DataFrame(results)


In [ ]:
# MAP ORIGIN CODES TO MEANINGFUL LABELS

origin_map = {
    '0': 'unknown',
    '1': 'germline',
    '2': 'somatic',
    '4': 'inherited',
    '8': 'paternal',
    '16': 'maternal',
    '32': 'de-novo',
    '64': 'biparental',
    '128': 'uniparental',
    '256': 'not-tested',
    '512': 'tested-inconclusive',
    '1073741824': 'other'
}

def extract_mc(info):
    if pd.isna(info):
        return []
    # Match MC=... until next ; or end of string
    match = re.search(r'MC=([^;]+)(?:;|$)', info)
    if match:
        mc_field = match.group(1)
        entries = mc_field.split(',')
        effects = [entry.split('|')[1] for entry in entries if '|' in entry]
        return effects
    return []

### Analysis

##### 1. Extract coordinates from all annottated NAGNAG 

In [ ]:
# genome annotation
gtf_df = pd.read_table(GTF_PATH,
                       names = ['SeqID', 'Source', 'Type', 'Start', 'End', 'Score', 'STR', 'Phase', 'Attributes'],
                       comment='#',
                       low_memory=False)

# get exon pairs
exon_df = extract_exon_pairs(gtf_df)

# get naganag coordinates
nagnag_df = extract_nagnag(exon_df)

# get +1 position at the 3' end
nagnag_df['Position_1_end'] = nagnag_df.apply(lambda row: row['shortIE']+1 if row['STR'] == '+' else row['shortIS']-1, axis=1)

nagnag_df

##### 2. Select desease related snv variants located at NAGNAG (+1 position)

In [ ]:
# read clinvar file
clinvar_df = pd.read_csv(CLINVAR_PATH, sep='\t', comment='#', low_memory=False, header=None, names=['CHROM','POS','ID','REF','ALT','QUAL','FILTER','INFO']) 
# filter only single_nucleotide_variant in info column
clinvar_df = clinvar_df[clinvar_df['INFO'].str.contains('single_nucleotide_variant', na=False)]
clinvar_df

In [ ]:
# merge clinvar_df with nagnag_df on chromosome and position
nagnag_var_df = nagnag_df.merge(clinvar_df, 
                    left_on=['SeqID', 'Position_1_end'], 
                    right_on=['CHROM', 'POS'], 
                    how='inner', 
                    suffixes=('', '_clinvar'))

nagnag_var_df

In [ ]:
# extract CLNSIG, ORIGIN and MC from INFO column
nagnag_var_df['CLNSIG'] = nagnag_var_df['INFO'].str.extract(r'CLNSIG=([^;]+)(?:;|$)')
nagnag_var_df['ORIGIN'] = nagnag_var_df['INFO'].str.extract(r'ORIGIN=([^;]+)(?:;|$)')[0].map(origin_map).fillna('nan')
nagnag_var_df['MC'] = nagnag_var_df['INFO'].apply(extract_mc)

# filter only pathogenic and likely pathogenic variants
nagnag_var_df = nagnag_var_df[nagnag_var_df['CLNSIG'].isin(['Pathogenic', 'Likely_pathogenic'])].copy()

nagnag_var_df

##### 3. Filter and save selected variants coordinates dataframe

In [ ]:
# change gene and chr col name
nagnag_var_df = nagnag_var_df.rename(columns={'Gene': 'GENE', 'SeqID': 'CHR'})
# keep only relevant columns
nagnag_var_df = nagnag_var_df[['GENE', 'CHR', 'STR', 
         'shortIS', 'shortIE', 'longIS', 'longIE',
         'POS', 'REF', 'ALT', 'CLNSIG', 'ORIGIN', 'MC', 'INFO']].copy()

nagnag_var_df


In [ ]:
# save to csv
nagnag_var_df.to_csv(f'{OUTPUT_DIR}/ClinVar_variants.csv', index=False)